In [2]:
!pip install shap 

   ---------------------------------------- 0.0/555.9 kB ? eta -:--:--
   ---------------------------------------- 555.9/555.9 kB 8.8 MB/s eta 0:00:00

   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   ---------------------------------------- 2/2 [shap]



In [9]:
# Data handling
import pandas as pd
import numpy as np

# ML
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report, ConfusionMatrixDisplay

# Explainability
import shap  # most important add-on — explains ML predictions

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Optional: Streamlit for the UI
import streamlit as st

print("Success")

Success


In [10]:
import yfinance as yf
import pandas as pd
import numpy as np
import time

# ── Nifty 100 tickers (a good starting universe) ────────────────────────────
NIFTY100 = [
    "RELIANCE.NS", "TCS.NS", "HDFCBANK.NS", "INFY.NS", "ICICIBANK.NS",
    "HINDUNILVR.NS", "ITC.NS", "KOTAKBANK.NS", "LT.NS", "AXISBANK.NS",
    "BAJFINANCE.NS", "ASIANPAINT.NS", "MARUTI.NS", "SUNPHARMA.NS",
    "TITAN.NS", "ULTRACEMCO.NS", "WIPRO.NS", "NESTLEIND.NS", "TECHM.NS",
    "HCLTECH.NS", "DIVISLAB.NS", "DRREDDY.NS", "CIPLA.NS", "APOLLOHOSP.NS",
    "TATAMOTORS.NS", "TATASTEEL.NS", "JSWSTEEL.NS", "COALINDIA.NS",
    "BAJAJFINSV.NS", "ADANIENT.NS", "ADANIPORTS.NS", "NTPC.NS",
    "POWERGRID.NS", "ONGC.NS", "BPCL.NS", "IOC.NS", "INDIGO.NS",
    "EICHERMOT.NS", "HEROMOTOCO.NS", "BAJAJ-AUTO.NS", "TATACONSUM.NS",
    "BRITANNIA.NS", "GODREJCP.NS", "DABUR.NS", "PIDILITIND.NS",
    "BERGEPAINT.NS", "HAVELLS.NS", "VOLTAS.NS", "GRASIM.NS", "AMBUJACEM.NS"
]


def safe_get(series, label):
    """Safely extract a value from a financial statement row."""
    try:
        if label in series.index:
            val = series.loc[label].iloc[0]
            return float(val) if pd.notna(val) else np.nan
        return np.nan
    except Exception:
        return np.nan


def fetch_company_data(ticker: str) -> dict | None:
    """
    Pull one year of financial data for a company.
    Returns a dict of raw financial items, or None if fetch fails.
    """
    try:
        t = yf.Ticker(ticker)
        info = t.info
        bs   = t.balance_sheet    # balance sheet
        inc  = t.income_stmt      # income statement
        cf   = t.cashflow         # cash flow statement

        # Need at least some data to proceed
        if bs.empty or inc.empty:
            print(f"  SKIP {ticker} — no financials")
            return None

        # ── Balance sheet items ─────────────────────────────────────────────
        total_assets       = safe_get(bs, "Total Assets")
        current_assets     = safe_get(bs, "Current Assets")
        current_liabilities= safe_get(bs, "Current Liabilities")
        total_liabilities  = safe_get(bs, "Total Liabilities Net Minority Interest")
        retained_earnings  = safe_get(bs, "Retained Earnings")
        stockholders_eq    = safe_get(bs, "Stockholders Equity")

        # Working capital
        working_capital = (current_assets - current_liabilities
                           if not np.isnan(current_assets) and not np.isnan(current_liabilities)
                           else np.nan)

        # ── Income statement items ──────────────────────────────────────────
        ebit        = safe_get(inc, "EBIT")
        revenue     = safe_get(inc, "Total Revenue")
        net_income  = safe_get(inc, "Net Income")

        # ── Market data ─────────────────────────────────────────────────────
        market_cap  = info.get("marketCap", np.nan)
        shares      = info.get("sharesOutstanding", np.nan)
        price       = info.get("currentPrice", np.nan)

        return {
            "ticker":             ticker,
            "company":            info.get("longName", ticker),
            "sector":             info.get("sector", "Unknown"),
            "total_assets":       total_assets,
            "working_capital":    working_capital,
            "retained_earnings":  retained_earnings,
            "ebit":               ebit,
            "revenue":            revenue,
            "net_income":         net_income,
            "total_liabilities":  total_liabilities,
            "book_equity":        stockholders_eq,
            "market_cap":         market_cap,
            "shares":             shares,
            "price":              price,
        }

    except Exception as e:
        print(f"  ERROR {ticker}: {e}")
        return None


def build_dataset(tickers: list = NIFTY100,
                  output_path: str = "raw_financials.csv") -> pd.DataFrame:
    """
    Loop through tickers, fetch data, return clean DataFrame.
    Adds a 1-second sleep between calls to be polite to Yahoo Finance.
    """
    records = []
    for i, ticker in enumerate(tickers):
        print(f"[{i+1}/{len(tickers)}] Fetching {ticker}...")
        result = fetch_company_data(ticker)
        if result:
            records.append(result)
        time.sleep(1)  # avoid rate limiting

    df = pd.DataFrame(records)
    df.to_csv(output_path, index=False)
    print(f"\nSaved {len(df)} companies to {output_path}")
    return df

print("Success")

Success


In [11]:
def compute_zprime_score(df: pd.DataFrame) -> pd.DataFrame:
    """
    Altman Z''-Score (1995 emerging market variant).
    Formula: Z'' = 6.56*X1 + 3.26*X2 + 6.72*X3 + 1.05*X4

    X1 = Working Capital / Total Assets        (liquidity)
    X2 = Retained Earnings / Total Assets      (profitability history)
    X3 = EBIT / Total Assets                   (operating efficiency)
    X4 = Book Equity / Total Liabilities       (leverage — uses BOOK value,
                                                 not market, for India)

    Cutoffs:  > 2.6  → Safe
              1.1–2.6 → Grey Zone
              < 1.1  → Distress
    """
    df = df.copy()

    # Guard against division by zero
    ta  = df["total_assets"].replace(0, np.nan)
    tl  = df["total_liabilities"].replace(0, np.nan)

    df["X1"] = df["working_capital"]    / ta
    df["X2"] = df["retained_earnings"]  / ta
    df["X3"] = df["ebit"]               / ta
    df["X4"] = df["book_equity"]        / tl

    df["zscore"] = (
        6.56 * df["X1"] +
        3.26 * df["X2"] +
        6.72 * df["X3"] +
        1.05 * df["X4"]
    )

    df["zone"] = pd.cut(
        df["zscore"],
        bins=[-np.inf, 1.1, 2.6, np.inf],
        labels=["Distress", "Grey Zone", "Safe"]
    )

    return df


def create_distress_label(df: pd.DataFrame) -> pd.DataFrame:
    """
    Relaxed labelling strategy for large-cap universe.
    Uses Z-Score zone directly as the label — Grey Zone or below = distressed.
    This gives a more balanced dataset when working with blue-chip companies.
    """
    df = df.copy()

    # Label: Grey Zone OR Distress zone = 1
    df["distress_flag"] = (df["zscore"] < 2.6).astype(int)

    print(f"Distress/Grey: {df['distress_flag'].sum()} | Healthy: {(df['distress_flag']==0).sum()}")
    return df

print("Success")

Success


In [12]:



FEATURES = ["X1", "X2", "X3", "X4"]


def prepare_data(df: pd.DataFrame):
    """Drop rows with NaN in features or target, return X and y."""
    clean = df.dropna(subset=FEATURES + ["distress_flag"])
    X = clean[FEATURES]
    y = clean["distress_flag"]
    print(f"Training on {len(clean)} companies | Class balance: {y.mean():.1%} distressed")
    return X, y, clean


def train_and_compare(X, y) -> dict:
    # Guard: need at least 2 classes to train
    if y.nunique() < 2:
        raise ValueError(
            f"Only one class found in y ({y.unique()}). "
            "Check your distress labelling — try relaxing the threshold."
        )

    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
        "Gradient Boosting":   GradientBoostingClassifier(n_estimators=200, max_depth=3,
                                                           learning_rate=0.05),
        "Random Forest":       RandomForestClassifier(n_estimators=300, class_weight="balanced"),
    }

    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Use min(5, minority_class_count) folds to avoid fold-has-one-class errors
    n_minority = y.value_counts().min()
    n_splits   = min(5, n_minority)
    print(f"Minority class count: {n_minority} → using {n_splits}-fold CV")

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    results = {}
    for name, model in models.items():
        scores = cross_val_score(model, X_scaled, y,
                                 cv=cv, scoring="roc_auc", n_jobs=-1)
        model.fit(X_scaled, y)
        results[name] = {
            "model":  model,
            "cv_auc": scores.mean(),
            "cv_std": scores.std(),
        }
        print(f"{name:30s}  AUC = {scores.mean():.3f} ± {scores.std():.3f}")

    return results, scaler


def explain_with_shap(model, X, feature_names=FEATURES):
    """
    SHAP summary plot — which ratio drives distress predictions most?
    Works with tree-based models (GBM, RF).
    For Logistic Regression, use LinearExplainer instead.
    """
    # Choose right explainer type
    if hasattr(model, "estimators_"):  # tree-based
        explainer = shap.TreeExplainer(model)
    else:                               # linear
        explainer = shap.LinearExplainer(model, X)

    shap_values = explainer.shap_values(X)

    # Handle binary classification output shape
    if isinstance(shap_values, list):
        shap_values = shap_values[1]  # take class=1 (distress)

    print("\nGenerating SHAP summary plot...")
    shap.summary_plot(
        shap_values, X,
        feature_names=feature_names,
        plot_type="bar",
        show=True
    )

    # Waterfall for one company (most distressed prediction)
    preds = model.predict_proba(X)[:, 1]
    most_at_risk_idx = preds.argmax()

    shap.waterfall_plot(
        shap.Explanation(
            values=shap_values[most_at_risk_idx],
            base_values=explainer.expected_value
                        if not isinstance(explainer.expected_value, list)
                        else explainer.expected_value[1],
            data=X.iloc[most_at_risk_idx].values,
            feature_names=feature_names
        )
    )
    return shap_values


def score_all_companies(df_clean, model, scaler, features=FEATURES) -> pd.DataFrame:
    """
    Add ML probability of default (PD) to full company DataFrame.
    This is the column you'll use in your Power BI dashboard.
    """
    X = df_clean[features].fillna(df_clean[features].median())
    X_scaled = scaler.transform(X)
    df_clean = df_clean.copy()
    df_clean["ml_pd_score"] = model.predict_proba(X_scaled)[:, 1]
    return df_clean

print("Success")

Success


In [13]:

from sklearn.model_selection import StratifiedKFold, cross_val_score
# ── Step 1: Fetch data ───────────────────────────────────────────────────────
print("=" * 50)
print("STEP 1: Fetching financial data from yfinance")
print("=" * 50)

# First run: fetches from Yahoo Finance and saves CSV
# Subsequent runs: load from CSV to avoid re-fetching
import os
CSV = "raw_financials.csv"

if os.path.exists(CSV):
    print(f"Loading cached data from {CSV}")
    df_raw = pd.read_csv(CSV)
else:
    df_raw = build_dataset(NIFTY100, output_path=CSV)

print(f"Loaded {len(df_raw)} companies")

# ── Step 2: Compute Altman Z''-Score ────────────────────────────────────────
print("\n" + "=" * 50)
print("STEP 2: Computing Altman Z''-Scores")
print("=" * 50)

df = compute_zprime_score(df_raw)
df = create_distress_label(df)

# Show zone distribution
print("\nZone distribution:")
print(df["zone"].value_counts())

# ── Step 3: Train ML models ──────────────────────────────────────────────────
print("\n" + "=" * 50)
print("STEP 3: Training ML models")
print("=" * 50)

X, y, df_clean = prepare_data(df)
results, scaler = train_and_compare(X, y)

# Pick best model
best_name = max(results, key=lambda k: results[k]["cv_auc"])
best_model = results[best_name]["model"]
print(f"\nBest model: {best_name}  (AUC = {results[best_name]['cv_auc']:.3f})")

# ── Step 4: SHAP explanation ─────────────────────────────────────────────────
print("\n" + "=" * 50)
print("STEP 4: SHAP Explainability")
print("=" * 50)

# Only tree-based models support TreeExplainer
if "Logistic" not in best_name:
    explain_with_shap(best_model, X)

# ── Step 5: Score all companies + export ────────────────────────────────────
print("\n" + "=" * 50)
print("STEP 5: Scoring all companies")
print("=" * 50)

df_scored = score_all_companies(df_clean, best_model, scaler)

# Final output columns for dashboard
output_cols = [
    "ticker", "company", "sector",
    "X1", "X2", "X3", "X4",
    "zscore", "zone",
    "net_income", "distress_flag",
    "ml_pd_score"
]

df_output = df_scored[output_cols].sort_values("ml_pd_score", ascending=False)
df_output.to_csv("scored_companies.csv", index=False)
df_output.to_excel("scored_companies.xlsx", index=False)

print(f"\nExported {len(df_output)} scored companies")
print("\nTop 10 highest ML PD score:")
print(df_output.head(10)[["company", "sector", "zscore", "zone", "ml_pd_score"]].to_string())

# ── Step 6: Disagreement analysis ───────────────────────────────────────────
print("\n" + "=" * 50)
print("STEP 6: Model Disagreement Analysis")
print("=" * 50)

# The interview-worthy finding: where Z-Score says Safe but ML says Risky
df_output["zscore_safe"] = df_output["zone"] == "Safe"
df_output["ml_risky"]    = df_output["ml_pd_score"] > 0.5

disagree = df_output[df_output["zscore_safe"] & df_output["ml_risky"]]
print(f"\nCompanies Z-Score calls SAFE but ML flags as HIGH RISK: {len(disagree)}")
print(disagree[["company", "sector", "zscore", "ml_pd_score"]].to_string())

STEP 1: Fetching financial data from yfinance
Loading cached data from raw_financials.csv
Loaded 49 companies

STEP 2: Computing Altman Z''-Scores
Distress/Grey: 12 | Healthy: 37

Zone distribution:
zone
Safe         30
Grey Zone     7
Distress      5
Name: count, dtype: int64

STEP 3: Training ML models
Training on 42 companies | Class balance: 28.6% distressed
Minority class count: 12 → using 5-fold CV
Logistic Regression             AUC = 1.000 ± 0.000
Gradient Boosting               AUC = 0.917 ± 0.075
Random Forest                   AUC = 1.000 ± 0.000

Best model: Logistic Regression  (AUC = 1.000)

STEP 4: SHAP Explainability

STEP 5: Scoring all companies

Exported 42 scored companies

Top 10 highest ML PD score:
                                    company           sector    zscore       zone  ml_pd_score
47                Grasim Industries Limited  Basic Materials  0.367067   Distress     0.978672
24                       Tata Steel Limited  Basic Materials  0.865554   Distre

In [8]:
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os

st.set_page_config(page_title="Credit Scoring Dashboard", layout="wide")
st.title("Credit Scoring — Altman Z'' + ML")

# Load scored data (run main.py first to generate this)
CSV = "scored_companies.csv"
if not os.path.exists(CSV):
    st.error("Run main.py first to generate scored_companies.csv")
    st.stop()

df = pd.read_csv(CSV)

# ── Top metrics ──────────────────────────────────────────────────────────────
c1, c2, c3, c4 = st.columns(4)
c1.metric("Companies Scored", len(df))
c2.metric("Distressed (Z'')", (df["zone"] == "Distress").sum())
c3.metric("Grey Zone",        (df["zone"] == "Grey Zone").sum())
c4.metric("High ML Risk (>50%)", (df["ml_pd_score"] > 0.5).sum())

st.divider()

# ── Main scatter: Z-Score vs ML PD ──────────────────────────────────────────
st.subheader("Z''-Score vs ML Probability of Default")
st.caption("The disagreement quadrant (top-right) is most interesting — safe by Z-Score, risky by ML.")

fig = px.scatter(
    df, x="zscore", y="ml_pd_score",
    color="zone",
    hover_data=["company", "sector", "X1", "X2", "X3", "X4"],
    color_discrete_map={
        "Safe":      "#22c55e",
        "Grey Zone": "#f59e0b",
        "Distress":  "#ef4444"
    },
    labels={"zscore": "Altman Z''-Score", "ml_pd_score": "ML Probability of Default"},
    height=500
)

# Add quadrant lines
fig.add_hline(y=0.5,  line_dash="dash", line_color="white", opacity=0.4)
fig.add_vline(x=2.6,  line_dash="dash", line_color="white", opacity=0.4)

# Annotation for disagreement quadrant
fig.add_annotation(x=3.5, y=0.75, text="⚠ Disagreement Zone",
                    font=dict(color="#f59e0b", size=11), showarrow=False)

fig.update_layout(template="plotly_dark")
st.plotly_chart(fig, use_container_width=True)

# ── Company table ────────────────────────────────────────────────────────────
st.subheader("All Companies — Ranked by ML Risk")

col1, col2 = st.columns([2, 1])
with col1:
    sector_filter = st.multiselect(
        "Filter by sector", options=df["sector"].unique(), default=df["sector"].unique()
    )
with col2:
    zone_filter = st.multiselect(
        "Filter by zone", options=["Safe", "Grey Zone", "Distress"],
        default=["Safe", "Grey Zone", "Distress"]
    )

filtered = df[df["sector"].isin(sector_filter) & df["zone"].isin(zone_filter)]
filtered_display = filtered[[
    "company", "sector", "zscore", "zone", "ml_pd_score",
    "X1", "X2", "X3", "X4"
]].sort_values("ml_pd_score", ascending=False)

st.dataframe(
    filtered_display.style.background_gradient(subset=["ml_pd_score"], cmap="RdYlGn_r"),
    hide_index=True,
    use_container_width=True
)

# ── Ratio breakdown by zone ──────────────────────────────────────────────────
st.subheader("Ratio Averages by Zone")
zone_avg = df.groupby("zone")[["X1", "X2", "X3", "X4"]].mean().reset_index()
fig2 = px.bar(
    zone_avg.melt(id_vars="zone", var_name="Ratio", value_name="Mean Value"),
    x="Ratio", y="Mean Value", color="zone", barmode="group",
    color_discrete_map={"Safe": "#22c55e", "Grey Zone": "#f59e0b", "Distress": "#ef4444"},
    template="plotly_dark", height=350
)
st.plotly_chart(fig2, use_container_width=True)

2026-04-05 13:25:41.787 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 13:25:41.790 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 13:25:41.792 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 13:25:41.819 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 13:25:41.820 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 13:25:41.823 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 13:25:41.825 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 13:25:41.826 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()